<a target="_blank" href="https://colab.research.google.com/github/TransformerLensOrg/TransformerLens/blob/main/demos/Boot_External_Demo.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Instrumenting a Pre-Loaded Model with `boot_external`

`TransformerBridge.boot_external(model=..., architecture=..., ...)` is the alternative for the cases the existing paths *don't* cover: when the model isn't an HF `PreTrainedModel` subclass. Custom research models, vLLM internals, hand-built `nn.Module` variants, models without a `.config` attribute, models you want to run with no tokenizer at all. `boot_external` decouples the bridge from HF's class hierarchy: it asks only for an `nn.Module` whose submodule tree matches the adapter's expected dot-paths, plus a config supplied explicitly.

**The honest coverage matrix:**

| Scenario | Use |
|---|---|
| HF model on Hub | `boot_transformers("name")` |
| HF-format directory on disk | `boot_transformers("/path/to/dir")` |
| Pre-loaded HF `PreTrainedModel` | `boot_transformers("name", hf_model=preloaded)` |
| `nn.Module` that isn't a `PreTrainedModel` | **`boot_external`** |
| Model with no HF config object | **`boot_external`** (with `tl_config=`) |
| Token-IDs-only inference (no tokenizer) | **`boot_external`** (with `tokenizer=None`) |

If your model is HF-shaped, `boot_transformers` already handles it. `boot_external` is for everything else. The bridge never moves, casts, or mutates the supplied model regardless of which entry point you use. This notebook uses a hand-built `nn.Module` so it runs anywhere (no Hub call, no GPU).

## Setup

In [19]:
# NBVAL_IGNORE_OUTPUT
import os

IN_GITHUB = os.getenv("GITHUB_ACTIONS") == "true"
try:
    import google.colab

    IN_COLAB = True
    print("Running as a Colab notebook")
except ImportError:
    IN_COLAB = False

if not IN_GITHUB and not IN_COLAB:
    print("Running as a Jupyter notebook - intended for development only!")
    from IPython import get_ipython

    ipython = get_ipython()
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")

if IN_COLAB or IN_GITHUB:
    %pip install transformer_lens

Running as a Jupyter notebook - intended for development only!
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
import torch
from transformers import AutoModelForCausalLM, GPT2Config

from transformer_lens.model_bridge import TransformerBridge

## 1. The bare-minimum call

`boot_external` needs three things:

1. **`model`**: your `nn.Module`. The bridge expects its submodule tree to match the architecture adapter's expected dot-paths. For `GPT2LMHeadModel`, that means `transformer.h[i].attn.c_attn`, `transformer.wte`, etc.
2. **`architecture`**: the string identifier for the adapter (same set as `boot_transformers`: `"GPT2LMHeadModel"`, `"LlamaForCausalLM"`, etc.).
3. **Config**: exactly one of `hf_config=` (HuggingFace-style config that gets translated) or `tl_config=` (a pre-built `TransformerBridgeConfig`).

Below we build a tiny 2-layer GPT-2-shaped module with random weights. In production this would be your already-loaded checkpoint.

In [21]:
# Imagine this `model` came from your own training loop, a quantization
# pipeline, or any non-HF loader. We use `from_config` here so the demo runs
# anywhere without a Hub download — the weights are random.
config = GPT2Config(
    n_layer=2,
    n_head=2,
    n_embd=16,
    n_positions=64,
    vocab_size=100,
)
model = AutoModelForCausalLM.from_config(config)
model.eval()

bridge = TransformerBridge.boot_external(
    model=model,
    architecture="GPT2LMHeadModel",
    hf_config=config,
    tokenizer=None,  # see section 3
)

print(f"n_layers={bridge.cfg.n_layers}, d_model={bridge.cfg.d_model}")
print(f"hook registry size: {len(bridge._hook_registry)}")

n_layers=2, d_model=16
hook registry size: 82


## 2. Running the model with hooks

Once wrapped, the bridge behaves the same as one built via `boot_transformers`. `run_with_cache` captures activations, hooks fire, ablation/patching all work.

In [12]:
tokens = torch.randint(0, config.vocab_size, (1, 8))

with torch.inference_mode():
    logits, cache = bridge.run_with_cache(tokens)

print(f"logits shape: {tuple(logits.shape)}")
print(f"cache size: {len(cache)} hooks")
print("sample hooks:", sorted(cache.keys())[:5])

logits shape: (1, 8, 100)
cache size: 95 hooks
sample hooks: ['blocks.0.attn.hook_attn_scores', 'blocks.0.attn.hook_hidden_states', 'blocks.0.attn.hook_in', 'blocks.0.attn.hook_k', 'blocks.0.attn.hook_out']


## 3. Tokenizer-less mode

Pass `tokenizer=None` when your pipeline doesn't have an HF-compatible tokenizer (e.g. you're using SentencePiece directly, or your tokenizer lives in another process). The bridge runs in **token-IDs-only mode**, `bridge(token_id_tensor)` works, but `bridge("a string")` raises because there's no tokenizer to convert it.

When you do have a tokenizer (any `PreTrainedTokenizerBase`), pass it through and string prompts work as usual.

In [13]:
# Token IDs: works
logits = bridge(tokens, return_type="logits")
print("token-id forward OK:", tuple(logits.shape))

# String input: raises (no tokenizer)
try:
    bridge("this will fail")
except AssertionError as e:
    print("string forward raises as expected:", e)

token-id forward OK: (1, 8, 100)
string forward raises as expected: Cannot use to_tokens without a tokenizer


## 4. Pre-flight: `diagnose_paths`

The architecture adapter walks a fixed set of dot-paths (e.g. `transformer.h.0.attn.c_attn`). If your model's tree doesn't match — common when porting from a non-HF format like vLLM (which fuses `q_proj`/`k_proj`/`v_proj` into a single `qkv_proj`) — bridge construction fails with an `AttributeError` somewhere mid-walk.

`TransformerBridge.diagnose_paths(model, architecture, hf_config=...)` is a **pre-flight** check: it walks the adapter's expected paths against your model's tree *without constructing the bridge* and returns `{"resolved": [...], "missing": [...]}`. Each entry is a `tl_name -> expected_remote_path` string — the adapter's notion of where it expects to find a submodule.

Reports are **per-model**: an expected path like `embed -> transformer.wte` lands in `resolved` against a model that has `transformer.wte`, and in `missing` against one that doesn't. Below we diagnose two different models, so don't be surprised when the same path strings appear in opposite-looking groups across the two reports — it's the same adapter expectation, evaluated against two different trees.

In [15]:
import torch.nn as nn

def summarize(label, report):
    n_ok, n_miss = len(report["resolved"]), len(report["missing"])
    verdict = "safe to wrap" if n_miss == 0 else "WILL FAIL during boot — fix tree first"
    print(f"=== {label} ===")
    print(f"  resolved: {n_ok}")
    print(f"  missing:  {n_miss}")
    print(f"  verdict:  {verdict}")
    if report["missing"]:
        print("  missing paths (adapter expects these; model lacks them):")
        for line in report["missing"]:
            print(f"    - {line}")
    print()

# DIAGNOSIS A: a model that already matches the adapter's expected tree.
summarize(
    "Diagnosis A — healthy HF GPT-2 model",
    TransformerBridge.diagnose_paths(model, "GPT2LMHeadModel", hf_config=config),
)

# DIAGNOSIS B: a deliberately-wrong tree. Has lm_head but is missing the
# entire transformer.* subtree the adapter expects.
class MismatchedTree(nn.Module):
    def __init__(self):
        super().__init__()
        self.lm_head = nn.Linear(8, 100)

summarize(
    "Diagnosis B — module missing the transformer subtree",
    TransformerBridge.diagnose_paths(MismatchedTree(), "GPT2LMHeadModel", hf_config=config),
)

=== Diagnosis A — healthy HF GPT-2 model ===
  resolved: 13
  missing:  0
  verdict:  safe to wrap

=== Diagnosis B — module missing the transformer subtree ===
  resolved: 1
  missing:  4
  verdict:  WILL FAIL during boot — fix tree first
  missing paths (adapter expects these; model lacks them):
    - embed -> transformer.wte
    - pos_embed -> transformer.wpe
    - blocks -> transformer.h
    - ln_final -> transformer.ln_f



## 5. `hf_config` vs `tl_config`

Two config paths:

- **`hf_config=...`** — pass an HF-style config object (the same object your model was constructed from). Internally, `boot_external` calls `map_default_transformer_lens_config(hf_config)` to translate field names (`n_embd` → `d_model`, `num_attention_heads` → `n_heads`, etc.). This is the convenient path if you already have an HF config.
- **`tl_config=...`** — pass a fully-constructed `TransformerBridgeConfig`. Skips the HF translation entirely. Use this when your model has no HF config (training-loop checkpoints, raw torch builds, etc.) or when you need to override a translated value.

If you pass both, `tl_config` wins (explicit override). If you pass neither, `boot_external` raises — the bridge can't infer `d_model`/`n_heads`/`n_layers` from the module alone.

In [16]:
# Same model, but configured via an explicit TL config (no HF config involved
# in the boot path — useful when your loader doesn't produce one).
from transformer_lens.config import TransformerBridgeConfig
from transformer_lens.model_bridge.sources.transformers import (
    map_default_transformer_lens_config,
)

tl_cfg_dict = map_default_transformer_lens_config(config).__dict__
tl_cfg = TransformerBridgeConfig.from_dict(dict(tl_cfg_dict))

bridge_tl = TransformerBridge.boot_external(
    model=model,
    architecture="GPT2LMHeadModel",
    tl_config=tl_cfg,
)
print(f"built via tl_config: n_layers={bridge_tl.cfg.n_layers}")

built via tl_config: n_layers=2


## 6. The case `boot_transformers` can't handle: a non-`PreTrainedModel` module

This is the primary purpose of `boot_external`. We construct an `nn.Module` that has the right *submodule names* for the GPT-2 adapter but isn't a `PreTrainedModel` subclass — so it has no `.config`, no `.save_pretrained()`, no `.from_pretrained()`. It's just torch.

`boot_transformers(hf_model=...)` can't accept this — it reaches into `hf_model.config` ([transformers.py:409](transformer_lens/model_bridge/sources/transformers.py#L409)), which doesn't exist. `boot_external` accepts the config separately, so it works.

In practice this is what custom research models, vLLM internals, and hand-built variants look like.

In [18]:
# Wrap our model in a bare nn.Module — strips the PreTrainedModel-ness.
# In real codebases this is what your own training-loop model would look
# like: a torch.nn.Module that happens to have the right submodule names.
import torch.nn as nn

class CustomModelWrapper(nn.Module):
    """A non-PreTrainedModel wrapper. No .config, no .save_pretrained."""
    def __init__(self, hf_model):
        super().__init__()
        # Mirror the submodule structure. The bridge walks these by name.
        self.transformer = hf_model.transformer
        self.lm_head = hf_model.lm_head

    def forward(self, input_ids, **kwargs):
        # Return raw logits — the bridge accepts a tensor, an object with
        # `.logits`, or a tuple where index 0 is logits (bridge.py:1932-1937).
        hidden = self.transformer(input_ids).last_hidden_state
        return self.lm_head(hidden)

source_model = AutoModelForCausalLM.from_config(config)
custom_model = CustomModelWrapper(source_model)
custom_model.eval()

# Verify it's no longer a PreTrainedModel:
from transformers import PreTrainedModel
print("is PreTrainedModel:", isinstance(custom_model, PreTrainedModel))
print("has .config:       ", hasattr(custom_model, "config"))

# boot_external still works — the config is supplied separately:
bridge_custom = TransformerBridge.boot_external(
    model=custom_model,
    architecture="GPT2LMHeadModel",
    hf_config=config,  # we still have it; it's just not on the model
)
print("\nbridge wired:", len(bridge_custom._hook_registry), "hooks")

# Sanity-check it runs:
with torch.inference_mode():
    logits = bridge_custom(torch.randint(0, 100, (1, 4)), return_type="logits")
print("forward shape:", tuple(logits.shape))

is PreTrainedModel: False
has .config:        False

bridge wired: 82 hooks
forward shape: (1, 4, 100)


## When `boot_external` is the wrong tool

- **Model name from HF Hub or local HF-format dir.** Use `boot_transformers("name")` or `boot_transformers("/path/")`. It requires fewer arguments and has automatic config translation + automatic tokenizer download.
- **HF `PreTrainedModel` already in memory.** Use `boot_transformers("name", hf_model=preloaded)`. It's been there a while and avoids the second download; `boot_external` doesn't add anything here.
- **In-process quantized models loaded via `from_pretrained(quantization_config=...)`.** These are still `PreTrainedModel` instances — `boot_transformers(hf_model=...)` works.
- **Module tree doesn't match the adapter.** vLLM, llama.cpp, and other inference engines fuse projections (e.g. one `qkv_proj` instead of three separate `q_proj`/`k_proj`/`v_proj`). `boot_external` won't auto-bridge those — you either need a custom adapter, or to transfer the state_dict into an HF-shaped shell first. The vLLM walkthrough covers this case.
- **Multimodal models needing an HF `AutoProcessor`.** The bridge's processor setup is HF-specific. With `boot_external` you'd need to set `bridge.processor = ...` manually.